### Understanding Docker Isolation

Containers have separate network namespaces, so `localhost` inside the frontend refers to the frontend container itself. Containers on Docker's default bridge can normally communicate using their container IP addresses, but they do not automatically resolve each other's container names. A user-defined bridge network provides this name resolution. We will put both containers on that network and tell the frontend the backend's container name and internal port.

See the [Docker bridge network documentation](https://docs.docker.com/engine/network/drivers/bridge/).

---

### Task: Connecting Multiple Containers Using Docker CLI

To connect the frontend and backend containers, you'll need to set up a Docker network that allows the containers to communicate. Here's how to do that step-by-step:

#### Step 1: Stop and Remove Any Running Containers

Before proceeding, **stop** and **remove** any containers you’ve already started for the frontend and backend, as running duplicate containers can cause conflicts:

```bash
docker stop frontend-container backend-container
docker rm frontend-container backend-container
```

#### Step 2: Create a Docker Network

Create a new network that both the frontend and backend containers can use to communicate with each other:

```bash
docker network create app-network
```

#### Step 3: Run the Backend Container

Run the backend container (`fhtw-ai-backend`) and attach it to the newly created network:

```bash
docker run -d --name backend-container --network app-network -p 8080:80 fhtw-ai-backend
```

- **`--network app-network`**: Connects the container to the `app-network`.
- Port `8080` is published on the host; the backend still listens on port `80` inside the network.

#### Step 4: Run the Frontend Container

Run the frontend container (`fhtw-ai-frontend`) and connect it to the same network:

```bash
docker run -d --name frontend-container --network app-network -p 80:8501 -e APP_BACKEND_URL=http://backend-container:80 fhtw-ai-frontend
```

- Both containers share the network. `APP_BACKEND_URL` tells the frontend where to send requests.
- `-p 80:8501` publishes Streamlit's internal port `8501` on host port `80`.

#### Step 5: Test the Communication

- Now, try accessing the frontend app at `http://localhost:80`.
- The frontend should communicate with the backend using the **backend container name**, e.g., `http://backend-container:80`.
- If the communication fails, think about why this might be happening. Do both containers have access to each other?

---

### Recap of Docker CLI Commands

- **Stop and remove running containers**:
  ```bash
  docker stop frontend-container backend-container
  docker rm frontend-container backend-container
  ```

- **Create a network**:
  ```bash
  docker network create app-network
  ```

- **Run the backend container**:
  ```bash
  docker run -d --name backend-container --network app-network -p 8080:80 fhtw-ai-backend
  ```

- **Run the frontend container**:
  ```bash
  docker run -d --name frontend-container --network app-network -p 80:8501 -e APP_BACKEND_URL=http://backend-container:80 fhtw-ai-frontend
  ```

---

### Next Step: Docker Compose

Managing networks manually works for small projects, but Docker Compose makes it easier to manage multi-container applications with shared networks. Now that you’ve seen how to connect containers using Docker CLI, we will move on to using **Docker Compose**, which simplifies this process significantly for larger setups.